In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Downloading the Data

In [ ]:
import kagglehub
import pandas as pd
import os

# Downloading the dataset
path = kagglehub.dataset_download('marius2303/ad-click-prediction-dataset')


print("Files in dataset:", os.listdir(path))


Files in dataset: ['ad_click_dataset.csv']


In [ ]:
df = pd.read_csv(os.path.join(path, "ad_click_dataset.csv"))

# Data Checking

In [ ]:
df.head(5)

,id,full_name,age,gender,device_type,ad_position,browsing_history,time_of_day,click
0,670,User670,22.0,NaN,Desktop,Top,Shopping,Afternoon,1
1,3044,User3044,NaN,Male,Desktop,Top,NaN,NaN,1
2,5912,User5912,41.0,Non-Binary,NaN,Side,Education,Night,1
3,5418,User5418,34.0,Male,NaN,NaN,Entertainment,Evening,1
4,9452,User9452,39.0,Non-Binary,NaN,NaN,Social Media,Morning,0


# DataFrame Summary

This dataset contains 10,000 user interaction records with ads.




In [ ]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                10000 non-null  int64  
 1   full_name         10000 non-null  object 
 2   age               5234 non-null   float64
 3   gender            5307 non-null   object 
 4   device_type       8000 non-null   object 
 5   ad_position       8000 non-null   object 
 6   browsing_history  5218 non-null   object 
 7   time_of_day       8000 non-null   object 
 8   click             10000 non-null  int64  
dtypes: float64(1), int64(2), object(6)
memory usage: 703.3+ KB
None


 - The two columns `"id"` and `"full_name"` are not useful for calculating the `proportions_ztest` so they are dropped.

In [ ]:
# The two columns "id" and "full_name" are not useful for calculating the proportions_ztest
df.drop(columns=['id','full_name'], inplace=True)


- We can see that the columns `"age", "gender"`, `"browsing_history"` have aroun 5000 null values so it is better to drop them.
- Also, the columns `"time_of_day"`, `"device_type"`, and `"ad_position"` have around 2,000 null values, so they are being replaced by their respective mode (i.e., the most frequently occurring value in each column) to handle missing data effectively.



In [ ]:
print(df.isnull().sum())


age                 4766
gender              4693
device_type         2000
ad_position         2000
browsing_history    4782
time_of_day         2000
click                  0
dtype: int64


# Data is Cleaned Now

In [ ]:
# Drop columns with more than 4000 missing values
cols_to_drop = ['age', 'gender', 'browsing_history']
df.drop(columns=cols_to_drop, inplace=True)

# Fill mode for columns with 2000 missing values
for col in ['device_type', 'ad_position', 'time_of_day']:
    mode_value = df[col].mode()[0]
    df[col].fillna(mode_value, inplace=True)

# Confirm cleanup
print("Cleaned DataFrame:")
print(df.isnull().sum())
print(df.info())


Cleaned DataFrame:
id             0
full_name      0
device_type    0
ad_position    0
time_of_day    0
click          0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           10000 non-null  int64 
 1   full_name    10000 non-null  object
 2   device_type  10000 non-null  object
 3   ad_position  10000 non-null  object
 4   time_of_day  10000 non-null  object
 5   click        10000 non-null  int64 
dtypes: int64(2), object(4)
memory usage: 468.9+ KB
None


### One-Hot Encoding Categorical Variables

The code performs one-hot encoding on the categorical columns: `"device_type"`, `"ad_position"`, and `"time_of_day"`. This process converts each category within these columns into separate binary (0/1) columns, making the data suitable. The transformed DataFrame is then displayed using `df.head()`.


In [ ]:
import pandas as pd


categorical_cols = ['device_type', 'ad_position', 'time_of_day']

df = pd.get_dummies(df, columns=categorical_cols)

-
df.head()


,id,full_name,click,device_type_Desktop,device_type_Mobile,device_type_Tablet,ad_position_Bottom,ad_position_Side,ad_position_Top,time_of_day_Afternoon,time_of_day_Evening,time_of_day_Morning,time_of_day_Night
0,670,User670,1,True,False,False,False,False,True,True,False,False,False
1,3044,User3044,1,True,False,False,False,False,True,False,False,True,False
2,5912,User5912,1,True,False,False,False,True,False,False,False,False,True
3,5418,User5418,1,True,False,False,True,False,False,False,True,False,False
4,9452,User9452,0,True,False,False,True,False,False,False,False,True,False


### Creating Groups Based on Ad Position

The dataset is split into two groups based on ad position:
- **Group A:** Rows where the ad was shown at the top (`ad_position_Top` = 1)
- **Group B:** Rows where the ad was shown at the bottom (`ad_position_Bottom` = 1)


In [ ]:
# Group A: ad_position = 1 (Top)
group_a = df[df['ad_position_Top'] == 1]

# Group B: ad_position = 1 (Bottom)
group_b = df[df['ad_position_Bottom'] == 1]


In [ ]:
from statsmodels.stats.proportion import proportions_ztest
import numpy as np

# Number of clicks (successes) in each group
count = np.array([group_a['click'].sum(), group_b['click'].sum()])
# Total number of users in each group
nobs = np.array([len(group_a), len(group_b)])

In [ ]:
print("Group A - Clicks:", count[0], "Total Users:", nobs[0])
print("Group B - Clicks:", count[1], "Total Users:", nobs[1])


Group A - Clicks: 1649 Total Users: 2597
Group B - Clicks: 3218 Total Users: 4817


In [ ]:
from statsmodels.stats.proportion import proportions_ztest


z_stat, p_value = proportions_ztest(count, nobs)
print("Z-statistic:", z_stat)
print("P-value:", p_value)

if p_value < 0.05:
    print("There is a significant difference in click rates between the two groups.")
else:
    print("There is no significant difference in click rates between the two groups.")

Z-statistic: -2.861975515593176
P-value: 0.004210094293323208
There is a significant difference in click rates between the two groups.


### Inference from Proportions Z-Test

The z-test compares the click-through rates between ads shown at the **top** (Group A) and the **bottom** (Group B) of the page.

- **Z-statistic:** -2.86  
- **P-value:** 0.0042

Since the p-value is less than 0.05, we reject the null hypothesis.  
**Inference:** There is a statistically significant difference in click rates between the two ad positions, indicating that ad position has an impact on user click behavior.


### 2-Sample Z-Test for Proportions

To test whether there is a significant difference between the proportions of two groups, we use the following hypotheses:

- **Null Hypothesis (H₀):** p₁ = p₂  
- **Alternative Hypothesis (H₁):** p₁ ≠ p₂

#### Z-Test Statistic Formula:

$$
z = \frac{\hat{p}_1 - \hat{p}_2}
{\sqrt{\hat{p}(1 - \hat{p}) \left( \frac{1}{n_1} + \frac{1}{n_2} \right)}}
$$

Where:
- $\hat{p}_1$ and $\hat{p}_2$ are the sample proportions for groups 1 and 2  
- $n_1$ and $n_2$ are the sample sizes  
- $\hat{p}$ is the **pooled proportion**, calculated as:

$$
\hat{p} = \frac{x_1 + x_2}{n_1 + n_2}
$$

Here, $x_1 $ and $ x_2$ are the number of successes (e.g., clicks) in each group.

#### P-Value:
- The p-value is then obtained from the **standard normal distribution** using the calculated \(z\)-value.
- A small p-value (typically < 0.05) indicates that the difference in proportions is statistically significant.


# Question 2





## Loading the Data

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import ks_2samp

# Load the CSV files
train_df = pd.read_csv('train.csv')
test1_df = pd.read_csv('test1.csv')
test2_df = pd.read_csv('test2.csv')

In [ ]:
train_df.head(10)

,Unnamed: 0,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,Unnamed: 15,Unnamed: 16
0,1849,26/05/2004,19.00.00,-200,1130.0,-200.0,"22,7",1368.0,-200.0,933.0,-200.0,1709.0,1269.0,"26,7","19,5","0,6754",NaN,NaN
1,2533,24/06/2004,07.00.00,"1,2",1030.0,-200.0,"6,9",851.0,102.0,824.0,68.0,1700.0,983.0,"21,9","57,0","1,4742",NaN,NaN
2,3047,15/07/2004,17.00.00,"3,2",1164.0,-200.0,"20,3",1306.0,259.0,648.0,198.0,1886.0,1218.0,"35,5","19,1","1,0888",NaN,NaN
3,805,13/04/2004,07.00.00,"3,9",1496.0,524.0,"19,1",1272.0,328.0,667.0,130.0,2011.0,1399.0,"11,0","64,2","0,8398",NaN,NaN
4,2962,12/07/2004,04.00.00,-200,780.0,-200.0,"1,8",568.0,24.0,1200.0,34.0,1331.0,501.0,"19,9","51,3","1,1803",NaN,NaN
5,561,03/04/2004,03.00.00,"0,9",1042.0,66.0,"3,8",697.0,-200.0,1056.0,-200.0,1410.0,965.0,"15,1","57,6","0,9796",NaN,NaN
6,1423,09/05/2004,01.00.00,-200,1068.0,-200.0,"9,0",938.0,-200.0,862.0,-200.0,1575.0,1057.0,"12,7","64,9","0,9506",NaN,NaN
7,1910,29/05/2004,08.00.00,"0,6",843.0,-200.0,"4,0",712.0,77.0,1303.0,77.0,1343.0,548.0,"18,9","42,1","0,9081",NaN,NaN
8,489,31/03/2004,03.00.00,"0,5",832.0,22.0,"1,0",496.0,-200.0,1613.0,-200.0,1066.0,381.0,"14,2","36,1","0,5823",NaN,NaN
9,3048,15/07/2004,18.00.00,"3,6",1124.0,-200.0,"20,5",1309.0,236.0,655.0,183.0,1845.0,1170.0,"33,7","20,0","1,0275",NaN,NaN


- Combining dataframes to do some pre-processing.

In [ ]:
# Combining dataframes to do some pre-processing

combined_df = pd.concat([train_df, test1_df, test2_df], axis=0, ignore_index=True)

- Converting object (string) data types to float by replacing commas (",") with dots ("."), then casting to float.
- According to the data description, values of `-200` represent missing data and are replaced with `NaN`.
- Dropping non-informative columns: `'Date'`, `'Time'`, `'Unnamed: 0'`, `'Unnamed: 15'`, and `'Unnamed: 16'`.


In [ ]:
# Converting object(string) to float
cols_to_fix = ['CO(GT)','C6H6(GT)', 'T', 'RH', 'AH']

for col in cols_to_fix:
    combined_df[col] = combined_df[col].astype(str).str.replace(',', '.', regex=False).astype(float)

# According to dataset description -200 suggests missing value
combined_df.replace(-200, np.nan, inplace=True)

# Removing non-useful and unnamed columns
combined_df.drop(columns=['Date','Time','Unnamed: 0','Unnamed: 15','Unnamed: 16'], inplace=True)

In [ ]:
print("Summary of combined_df:")
combined_df.info()

Summary of combined_df:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4800 entries, 0 to 4799
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   CO(GT)         3928 non-null   float64
 1   PT08.S1(CO)    4715 non-null   float64
 2   NMHC(GT)       914 non-null    float64
 3   C6H6(GT)       4715 non-null   float64
 4   PT08.S2(NMHC)  4715 non-null   float64
 5   NOx(GT)        4118 non-null   float64
 6   PT08.S3(NOx)   4715 non-null   float64
 7   NO2(GT)        4115 non-null   float64
 8   PT08.S4(NO2)   4715 non-null   float64
 9   PT08.S5(O3)    4715 non-null   float64
 10  T              4715 non-null   float64
 11  RH             4715 non-null   float64
 12  AH             4715 non-null   float64
dtypes: float64(13)
memory usage: 487.6 KB


- Dropping the NMHC(GT) column because it has too many missing values
- For the remaining numeric columns, the missing values are filled with the median.
- Using the **median** helps avoid the influence of outliers.



In [ ]:
# Dropping the NMHC(GT) column because it has too many missing values
combined_df.drop(columns='NMHC(GT)', inplace=True)


numeric_cols = combined_df.select_dtypes(include=['number']).columns
for col in numeric_cols:
    median_val = combined_df[col].median()
    combined_df[col].fillna(median_val, inplace=True)

In [ ]:
train_rows = len(train_df)
test1_rows = len(test1_df)
test2_rows = len(test2_df)

# Agian split using slicing
train_df = combined_df.iloc[:train_rows].copy()
test1_df = combined_df.iloc[train_rows:train_rows + test1_rows].copy()
test2_df = combined_df.iloc[train_rows + test1_rows:].copy()


In [ ]:
print("Summary of train_df:")
print("-" * 50)
train_df.info()
print("\n" + "-" * 50 + "\n")

print("Summary of test1_df:")
print("-" * 50)
test1_df.info()
print("\n" + "-" * 50 + "\n")

print("Summary of test2_df:")
print("-" * 50)
test2_df.info()
print("\n" + "-" * 50)


Summary of train_df:
--------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3200 entries, 0 to 3199
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   CO(GT)         3200 non-null   float64
 1   PT08.S1(CO)    3200 non-null   float64
 2   C6H6(GT)       3200 non-null   float64
 3   PT08.S2(NMHC)  3200 non-null   float64
 4   NOx(GT)        3200 non-null   float64
 5   PT08.S3(NOx)   3200 non-null   float64
 6   NO2(GT)        3200 non-null   float64
 7   PT08.S4(NO2)   3200 non-null   float64
 8   PT08.S5(O3)    3200 non-null   float64
 9   T              3200 non-null   float64
 10  RH             3200 non-null   float64
 11  AH             3200 non-null   float64
dtypes: float64(12)
memory usage: 300.1 KB

--------------------------------------------------

Summary of test1_df:
--------------------------------------------------
<class 'pandas.core.frame.DataFrame'>

In [ ]:
print("Checking for missing values in train_df:")
print(train_df.isnull().sum())
print("\n" + "-"*50 + "\n")

print("Checking for missing values in test1_df:")
print(test1_df.isnull().sum())
print("\n" + "-"*50 + "\n")

print("Checking for missing values in test2_df:")
print(test2_df.isnull().sum())
print("\n" + "-"*50)

Checking for missing values in train_df:
CO(GT)           0
PT08.S1(CO)      0
C6H6(GT)         0
PT08.S2(NMHC)    0
NOx(GT)          0
PT08.S3(NOx)     0
NO2(GT)          0
PT08.S4(NO2)     0
PT08.S5(O3)      0
T                0
RH               0
AH               0
dtype: int64

--------------------------------------------------

Checking for missing values in test1_df:
CO(GT)           0
PT08.S1(CO)      0
C6H6(GT)         0
PT08.S2(NMHC)    0
NOx(GT)          0
PT08.S3(NOx)     0
NO2(GT)          0
PT08.S4(NO2)     0
PT08.S5(O3)      0
T                0
RH               0
AH               0
dtype: int64

--------------------------------------------------

Checking for missing values in test2_df:
CO(GT)           0
PT08.S1(CO)      0
C6H6(GT)         0
PT08.S2(NMHC)    0
NOx(GT)          0
PT08.S3(NOx)     0
NO2(GT)          0
PT08.S4(NO2)     0
PT08.S5(O3)      0
T                0
RH               0
AH               0
dtype: int64

-----------------------------------------------

# Data is clean now

In [ ]:
# Get all numeric columns common to all datasets
numeric_cols = list(set(train_df.select_dtypes(include='number').columns)
                    & set(test1_df.select_dtypes(include='number').columns)
                    & set(test2_df.select_dtypes(include='number').columns))

# KS test results storage
ks_results_test1 = {}
ks_results_test2 = {}

for col in numeric_cols:
    train_col = train_df[col]
    test1_col = test1_df[col]
    test2_col = test2_df[col]

    # KS test
    ks1 = ks_2samp(train_col, test1_col)
    ks2 = ks_2samp(train_col, test2_col)

    ks_results_test1[col] = (ks1.statistic, ks1.pvalue)
    ks_results_test2[col] = (ks2.statistic, ks2.pvalue)


## Kolmogorov-Smirnov (KS) Test: Calculating Statistic and p-value

The KS test compares the distributions of a feature in the training and test sets to detect covariate shift.

**KS Statistic (D):**

$$
D = \sup_x \left| F_1(x) - F_2(x) \right|
$$

- $ F_1(x) $: Empirical cumulative distribution function (CDF) of the feature in the training set
- $F_2(x) $: Empirical CDF of the feature in the test set
- $ \sup_x $: Maximum difference over all values of \( x \)

**p-value:**

- The p-value is computed based on the KS statistic \( D \) and the sizes of the two samples.
- It represents the probability of observing a KS statistic as extreme as \( D \) under the null hypothesis (that both samples are drawn from the same distribution).


- **KS Statistic (D):** Measures the largest difference between the two CDFs.
- **p-value:** Indicates if the distributions are significantly different (small p-value means significant difference).

In [ ]:

print("\nKS Test Results: Test1 vs Train")
print("-" * 50)
for col, (stat, pval) in ks_results_test1.items():
    print(f"{col:<20} | KS Statistic = {stat:8.4f} | p-value = {pval:8.4f}")

print("\nKS Test Results: Test2 vs Train")
print("-" * 50)
for col, (stat, pval) in ks_results_test2.items():
    print(f"{col:<20} | KS Statistic = {stat:8.4f} | p-value = {pval:8.4f}")



KS Test Results: Test1 vs Train
--------------------------------------------------
RH                   | KS Statistic =   0.0191 | p-value =   0.9722
PT08.S1(CO)          | KS Statistic =   0.0328 | p-value =   0.4900
C6H6(GT)             | KS Statistic =   0.0228 | p-value =   0.8885
PT08.S2(NMHC)        | KS Statistic =   0.0234 | p-value =   0.8686
NO2(GT)              | KS Statistic =   0.0166 | p-value =   0.9940
PT08.S5(O3)          | KS Statistic =   0.0300 | p-value =   0.6057
AH                   | KS Statistic =   0.0259 | p-value =   0.7764
PT08.S4(NO2)         | KS Statistic =   0.0219 | p-value =   0.9154
NOx(GT)              | KS Statistic =   0.0175 | p-value =   0.9885
T                    | KS Statistic =   0.0184 | p-value =   0.9799
CO(GT)               | KS Statistic =   0.0259 | p-value =   0.7764
PT08.S3(NOx)         | KS Statistic =   0.0344 | p-value =   0.4304

KS Test Results: Test2 vs Train
--------------------------------------------------
RH              

### Threshold for Significance

- A **p-value < 0.05** typically indicates a statistically significant difference in distributions (i.e., potential covariate shift).
- A **higher KS statistic** indicates a greater difference between the empirical CDFs of the train and test data.



### Test1 vs Train

Most features have:
- **KS statistic < 0.04**
- **p-value > 0.4** (many close to 1)

 **Interpretation:** The distributions of features in **test1** are **very similar** to those in the training set. Hence, **no significant covariate shift** is observed.

 ### Test2 vs Train

All features have:
- **KS statistic > 0.1**, some as high as **0.6**
- **p-value = 0.0000** (highly significant)

 **Interpretation:** The distributions of **test2** differ significantly from the training set across all features. This strongly suggests a **covariate shift** between **test2** and the training data.



###  Final Conclusion

- **Test1** does **not** exhibit covariate shift (high p-values, low KS stats).
- **Test2** **does** exhibit covariate shift (low p-values, high KS stats).

**Therefore, `test2.csv` exhibits covariate shift relative to `train.csv`.**


In [ ]:
# Covariate Shift Analysis
def covariate_shift_detection(ks_dict, alpha=0.05):
    shifted_cols = [col for col, (stat, pval) in ks_dict.items() if pval < alpha]
    return shifted_cols

shifted_test1 = covariate_shift_detection(ks_results_test1)
shifted_test2 = covariate_shift_detection(ks_results_test2)

print("\nCovariate Shift Analysis:")
if shifted_test1 and not shifted_test2:
    print("Covariate shift detected in test1.csv based on features:", shifted_test1)
elif shifted_test2 and not shifted_test1:
    print("Covariate shift detected in test2.csv based on features:", shifted_test2)
elif shifted_test1 and shifted_test2:
    print("Covariate shift detected in both test1.csv and test2.csv.")
    print("Test1 shifted features:", shifted_test1)
    print("Test2 shifted features:", shifted_test2)
else:
    print("No covariate shift detected in either test dataset.")



Covariate Shift Analysis:
Covariate shift detected in test2.csv based on features: ['RH', 'PT08.S1(CO)', 'C6H6(GT)', 'PT08.S2(NMHC)', 'NO2(GT)', 'PT08.S5(O3)', 'AH', 'PT08.S4(NO2)', 'NOx(GT)', 'T', 'CO(GT)', 'PT08.S3(NOx)']
